# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/likithagarlapati7-oss/flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 116 (delta 30), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 1.85 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [4]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [6]:
import pandas as pd

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [8]:
feature_df = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "search_volume",
        "content_type"
    ]
].copy()

In [9]:
feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article


In [10]:
feature_df.isna().sum()

,0
content_id,0
impressions_90d,0
clicks_90d,0
ctr,0
content_age_days,0
days_since_last_update,0
search_volume,2468
content_type,0


In [11]:
feature_df["search_volume"] = (
    feature_df["search_volume"]
    .fillna(0)
)

In [12]:
feature_df.isna().sum()

,0
content_id,0
impressions_90d,0
clicks_90d,0
ctr,0
content_age_days,0
days_since_last_update,0
search_volume,0
content_type,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
## My rule and its reason codes

### Rule:

# Rank pages for refresh priority by combining two signals:

# - Pages with low CTR despite having search visibility.
# - Pages with high staleness based on days since last update.

# The final refresh opportunity score is:

# Refresh Score =
# 0.5 × Low CTR Score +
# 0.5 × Staleness Score

# Pages with higher scores are placed higher in the refresh queue.

# ### Reason Code:

# LOW_CTR_STALE

# Explanation:
# The page receives this reason code when it has both weak click engagement and older content freshness signals. These pages are considered potential candidates for content refresh.

In [13]:
feature_df["low_ctr_score"] = (
    1 - feature_df["ctr"].rank(pct=True)
)

feature_df["stale_score"] = (
    feature_df["days_since_last_update"]
    .rank(pct=True)
)

feature_df["baseline_score"] = (
    0.5 * feature_df["low_ctr_score"]
    +
    0.5 * feature_df["stale_score"]
)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
feature_df["low_ctr_score"] = (
    1 - feature_df["ctr"].rank(pct=True)
)

feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type,low_ctr_score,stale_score,baseline_score,reason_code,action
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article,0.082300,0.336000,0.209150,LOW_CTR_STALE,REFRESH
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article,0.524883,0.655033,0.589958,LOW_CTR_STALE,REFRESH
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article,0.472150,0.336000,0.404075,LOW_CTR_STALE,REFRESH
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article,0.143583,0.588283,0.365933,LOW_CTR_STALE,REFRESH
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article,0.414733,0.122700,0.268717,LOW_CTR_STALE,REFRESH


In [21]:
feature_df["stale_score"] = (
    feature_df["days_since_last_update"]
    .rank(pct=True)
)

feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type,low_ctr_score,stale_score,baseline_score,reason_code,action
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article,0.082300,0.336000,0.209150,LOW_CTR_STALE,REFRESH
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article,0.524883,0.655033,0.589958,LOW_CTR_STALE,REFRESH
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article,0.472150,0.336000,0.404075,LOW_CTR_STALE,REFRESH
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article,0.143583,0.588283,0.365933,LOW_CTR_STALE,REFRESH
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article,0.414733,0.122700,0.268717,LOW_CTR_STALE,REFRESH


In [22]:
feature_df["baseline_score"] = (
    0.5 * feature_df["low_ctr_score"]
    +
    0.5 * feature_df["stale_score"]
)

feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type,low_ctr_score,stale_score,baseline_score,reason_code,action
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article,0.082300,0.336000,0.209150,LOW_CTR_STALE,REFRESH
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article,0.524883,0.655033,0.589958,LOW_CTR_STALE,REFRESH
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article,0.472150,0.336000,0.404075,LOW_CTR_STALE,REFRESH
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article,0.143583,0.588283,0.365933,LOW_CTR_STALE,REFRESH
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article,0.414733,0.122700,0.268717,LOW_CTR_STALE,REFRESH


In [23]:
feature_df["reason_code"] = "LOW_CTR_STALE"

feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type,low_ctr_score,stale_score,baseline_score,reason_code,action
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article,0.082300,0.336000,0.209150,LOW_CTR_STALE,REFRESH
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article,0.524883,0.655033,0.589958,LOW_CTR_STALE,REFRESH
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article,0.472150,0.336000,0.404075,LOW_CTR_STALE,REFRESH
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article,0.143583,0.588283,0.365933,LOW_CTR_STALE,REFRESH
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article,0.414733,0.122700,0.268717,LOW_CTR_STALE,REFRESH


In [24]:
feature_df["action"] = "REFRESH"

feature_df.head()

,content_id,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update,search_volume,content_type,low_ctr_score,stale_score,baseline_score,reason_code,action
0,content_304f48230142,3803,29,0.76,187,20,10.0,keyword article,0.082300,0.336000,0.209150,LOW_CTR_STALE,REFRESH
1,content_a1fb4e703a9e,15320,7,0.05,445,25,90.0,keyword article,0.524883,0.655033,0.589958,LOW_CTR_STALE,REFRESH
2,content_9aa793d4d895,12581,11,0.09,141,20,0.0,keyword article,0.472150,0.336000,0.404075,LOW_CTR_STALE,REFRESH
3,content_331d6c4de07b,11751,58,0.49,463,22,10.0,keyword article,0.143583,0.588283,0.365933,LOW_CTR_STALE,REFRESH
4,content_d99b7a2d90ca,19140,24,0.13,263,14,0.0,keyword article,0.414733,0.122700,0.268717,LOW_CTR_STALE,REFRESH


In [25]:
baseline_queue = feature_df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].sort_values(
    by="baseline_score",
    ascending=False
)

In [26]:
baseline_queue.head(10)

,content_id,baseline_score,reason_code,action
29384,content_f6fdf87348f6,0.889875,LOW_CTR_STALE,REFRESH
26242,content_55a5b1c46474,0.889875,LOW_CTR_STALE,REFRESH
18440,content_8d56efff1e71,0.889833,LOW_CTR_STALE,REFRESH
24216,content_1b4ec72dafd4,0.889833,LOW_CTR_STALE,REFRESH
15608,content_06e19c6486b0,0.889783,LOW_CTR_STALE,REFRESH
8631,content_e2b702f4f92b,0.889783,LOW_CTR_STALE,REFRESH
7509,content_7a888d3d99c8,0.889733,LOW_CTR_STALE,REFRESH
18841,content_94991fe6268c,0.889733,LOW_CTR_STALE,REFRESH
21984,content_02b0d6e30129,0.889733,LOW_CTR_STALE,REFRESH
15790,content_6476d1d8c050,0.889733,LOW_CTR_STALE,REFRESH


In [27]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

In [28]:
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

In [29]:
pd.read_csv(
    "work/outputs/baseline_action_score.csv"
).head(10)

,content_id,baseline_score,reason_code,action
0,content_f6fdf87348f6,0.889875,LOW_CTR_STALE,REFRESH
1,content_55a5b1c46474,0.889875,LOW_CTR_STALE,REFRESH
2,content_8d56efff1e71,0.889833,LOW_CTR_STALE,REFRESH
3,content_1b4ec72dafd4,0.889833,LOW_CTR_STALE,REFRESH
4,content_06e19c6486b0,0.889783,LOW_CTR_STALE,REFRESH
5,content_e2b702f4f92b,0.889783,LOW_CTR_STALE,REFRESH
6,content_7a888d3d99c8,0.889733,LOW_CTR_STALE,REFRESH
7,content_94991fe6268c,0.889733,LOW_CTR_STALE,REFRESH
8,content_02b0d6e30129,0.889733,LOW_CTR_STALE,REFRESH
9,content_6476d1d8c050,0.889733,LOW_CTR_STALE,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [30]:
baseline_queue.head(20)

,content_id,baseline_score,reason_code,action
29384,content_f6fdf87348f6,0.889875,LOW_CTR_STALE,REFRESH
26242,content_55a5b1c46474,0.889875,LOW_CTR_STALE,REFRESH
18440,content_8d56efff1e71,0.889833,LOW_CTR_STALE,REFRESH
24216,content_1b4ec72dafd4,0.889833,LOW_CTR_STALE,REFRESH
15608,content_06e19c6486b0,0.889783,LOW_CTR_STALE,REFRESH
8631,content_e2b702f4f92b,0.889783,LOW_CTR_STALE,REFRESH
7509,content_7a888d3d99c8,0.889733,LOW_CTR_STALE,REFRESH
18841,content_94991fe6268c,0.889733,LOW_CTR_STALE,REFRESH
21984,content_02b0d6e30129,0.889733,LOW_CTR_STALE,REFRESH
15790,content_6476d1d8c050,0.889733,LOW_CTR_STALE,REFRESH


In [34]:
top20 = baseline_queue.head(20).copy()

top20

,content_id,baseline_score,reason_code,action
29384,content_f6fdf87348f6,0.889875,LOW_CTR_STALE,REFRESH
26242,content_55a5b1c46474,0.889875,LOW_CTR_STALE,REFRESH
18440,content_8d56efff1e71,0.889833,LOW_CTR_STALE,REFRESH
24216,content_1b4ec72dafd4,0.889833,LOW_CTR_STALE,REFRESH
15608,content_06e19c6486b0,0.889783,LOW_CTR_STALE,REFRESH
8631,content_e2b702f4f92b,0.889783,LOW_CTR_STALE,REFRESH
7509,content_7a888d3d99c8,0.889733,LOW_CTR_STALE,REFRESH
18841,content_94991fe6268c,0.889733,LOW_CTR_STALE,REFRESH
21984,content_02b0d6e30129,0.889733,LOW_CTR_STALE,REFRESH
15790,content_6476d1d8c050,0.889733,LOW_CTR_STALE,REFRESH


In [35]:
top20["confidence_note"] = (
    "High confidence because the page shows "
    "low CTR and content staleness signals."
)

top20["what_would_make_it_wrong"] = (
    "The recommendation may be wrong if the page "
    "has intentionally low CTR due to search intent "
    "or if freshness is not the actual issue."
)

In [36]:
top20[
    [
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
29384,content_f6fdf87348f6,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
26242,content_55a5b1c46474,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
18440,content_8d56efff1e71,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
24216,content_1b4ec72dafd4,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
15608,content_06e19c6486b0,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
8631,content_e2b702f4f92b,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
7509,content_7a888d3d99c8,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
18841,content_94991fe6268c,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
21984,content_02b0d6e30129,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...
15790,content_6476d1d8c050,REFRESH,LOW_CTR_STALE,High confidence because the page shows low CTR...,The recommendation may be wrong if the page ha...


In [ ]:
# ## Top-20 Review

# The top 20 pages from the baseline ranking were manually reviewed.

# Each page receives the action REFRESH because the rule identifies pages with low CTR and high staleness.

# The confidence note explains why the rule selected the page, while the "what would make it wrong" field highlights possible false positives.

# These recommendations are decision-support only and do not prove that refreshing a page will improve performance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# ## Weak Picks + Leakage Check

# ### Weak Picks / Possible False Positives

# The baseline rule ranks pages based on low CTR and content staleness. However, some top-ranked pages may not actually require a refresh.

# Possible weak picks include:

# 1. Pages with naturally low CTR:
#    - Some pages may receive impressions but fewer clicks because users get the required information directly from the search result.
#    - Low CTR does not always indicate poor content quality.

# 2. Old pages that still perform well:
#    - A page may have not been updated recently but may still satisfy users and generate consistent traffic.
#    - Age alone does not guarantee that content needs improvement.

# 3. Pages with changing search intent:
#    - A decrease in clicks may happen because user behaviour or search trends have changed, not because the content is outdated.

# 4. Pages with limited search demand:
#    - Some pages may receive low engagement because the topic has low demand rather than because the content quality is poor.

# Therefore, the baseline ranking should be treated as a prioritization list for review and not as a guaranteed refresh requirement.

# ---

# ### Leakage Check

# The baseline rule only uses historical information available at the decision time.

# Checks performed:

# ✓ No future performance data was used.

# ✓ No post-refresh metrics were used.

# ✓ No product flags or existing FlyRank refresh labels were used as input features.

# ✓ The score only uses:
# - CTR
# - days_since_last_update

# which are available before making a refresh decision.

# ✓ No future windows, such as later clicks, later impressions, or after-refresh outcomes, were included.

# The baseline score is therefore a decision-support rule and does not use leaked information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.